# Preprocesamiento de Datos — Monitoreo Lechería Paillaco

Análisis basado en los **pilares del preprocesamiento**:
1. Identificación y exploración inicial
2. Detección de valores nulos / faltantes
3. Detección de duplicados
4. Detección de inconsistencias y anomalías
5. Estandarización y normalización
6. Limpieza y datos listos para análisis

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

---
## 1. Identificación y Exploración Inicial

In [ ]:
# Carga de datos
df = pd.read_csv('monitoreo_lecheria_paillaco.csv', na_values=['NULL', 'null', 'NaN', ''])
print(f'Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')

In [ ]:
# Primeros registros
df.head(10)

In [ ]:
# Tipos de datos e información general
df.info()

In [ ]:
# Estadísticas descriptivas de columnas numéricas
df.describe()

In [ ]:
# Valores únicos por columna
for col in df.columns:
    print(f'{col}: {df[col].nunique()} valores únicos')
    if df[col].nunique() <= 20:
        print(f'  -> {df[col].unique()}')
    print()

---
## 2. Detección de Valores Nulos / Faltantes

In [ ]:
# Conteo de nulos por columna
nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
})
print('=== Resumen de valores nulos ===')
print(resumen_nulos)
print(f'\nTotal de celdas nulas: {df.isnull().sum().sum()}')
print(f'Porcentaje total de nulos: {(df.isnull().sum().sum() / df.size * 100):.2f}%')

In [ ]:
# Filas con al menos un valor nulo
filas_con_nulos = df[df.isnull().any(axis=1)]
print(f'Filas con al menos un valor nulo: {len(filas_con_nulos)}')
filas_con_nulos

---
## 3. Detección de Duplicados

In [ ]:
# Filas completamente duplicadas
duplicados = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
print(f'Filas duplicadas (todas las ocurrencias): {len(duplicados)}')
print(f'Grupos de duplicados: {len(duplicados) - df.duplicated(keep="first").sum()}')
print(f'Filas a eliminar (keep=first): {df.duplicated(keep="first").sum()}')
print()
duplicados

---
## 4. Detección de Inconsistencias y Anomalías

### 4.1 Inconsistencia en formato de fechas (`Fecha_Control`)

In [ ]:
# Clasificar los formatos de fecha detectados
import re

def clasificar_formato_fecha(fecha):
    if pd.isna(fecha):
        return 'NaN'
    fecha = str(fecha).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', fecha):
        return 'YYYY-MM-DD (ISO)'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', fecha):
        return 'DD/MM/YYYY'
    elif re.match(r'^\d{2}-\d{2}-\d{4}$', fecha):
        return 'MM-DD-YYYY'
    elif re.match(r'^[A-Za-z]', fecha):
        return 'Texto (ej: Marzo 26)'
    else:
        return f'Otro: {fecha}'

df['_formato_fecha'] = df['Fecha_Control'].apply(clasificar_formato_fecha)
print('=== Formatos de fecha detectados ===')
print(df['_formato_fecha'].value_counts())
print()

# Ejemplos de cada formato
for fmt in df['_formato_fecha'].unique():
    ejemplo = df[df['_formato_fecha'] == fmt]['Fecha_Control'].iloc[0]
    print(f'  {fmt}: "{ejemplo}"')

### 4.2 Inconsistencia en `Estado_Salud` (capitalización)

In [ ]:
# Variantes detectadas en Estado_Salud
print('=== Variantes de Estado_Salud ===')
print(df['Estado_Salud'].value_counts())
print(f'\nTotal de variantes distintas: {df["Estado_Salud"].nunique()}')
print(f'\nVariantes únicas (case-insensitive): {df["Estado_Salud"].str.lower().str.strip().nunique()}')

### 4.3 Valores negativos en `Litros_Leche`

In [ ]:
# Litros de leche negativos (imposible físicamente)
negativos_leche = df[df['Litros_Leche'] < 0]
print(f'Registros con litros de leche negativos: {len(negativos_leche)}')
print(f'Rango de valores negativos: [{negativos_leche["Litros_Leche"].min()}, {negativos_leche["Litros_Leche"].max()}]')
negativos_leche[['ID_Vaca', 'Fecha_Control', 'Litros_Leche']]

### 4.4 Temperaturas corporales fuera de rango (`Temp_Corporal`)

In [ ]:
# Rango normal de temperatura corporal bovina: ~37.5 - 39.5 °C
temp_anomalas = df[(df['Temp_Corporal'] < 37.0) | (df['Temp_Corporal'] > 40.0)]
print(f'Registros con temperatura fuera de rango normal (37.0 - 40.0 °C): {len(temp_anomalas)}')
print(f'Rango completo en dataset: [{df["Temp_Corporal"].min()}, {df["Temp_Corporal"].max()}]')
print()

# Temperaturas claramente erróneas (>45°C o <35°C)
temp_erroneas = df[(df['Temp_Corporal'] > 45.0) | (df['Temp_Corporal'] < 35.0)]
print(f'Temperaturas claramente erróneas (>45°C): {len(temp_erroneas)}')
temp_erroneas[['ID_Vaca', 'Fecha_Control', 'Temp_Corporal']]

---
## 5. Estandarización y Normalización

In [ ]:
# Trabajar sobre una copia
df_clean = df.drop(columns=['_formato_fecha']).copy()
print(f'Registros iniciales: {len(df_clean)}')

### 5.1 Estandarizar `Estado_Salud`

In [ ]:
# Normalizar a formato Title Case y trim
df_clean['Estado_Salud'] = df_clean['Estado_Salud'].str.strip().str.title()

print('=== Estado_Salud después de estandarizar ===')
print(df_clean['Estado_Salud'].value_counts())

### 5.2 Estandarizar `Fecha_Control`

In [ ]:
def parsear_fecha(fecha):
    """Intenta parsear la fecha a formato datetime."""
    if pd.isna(fecha):
        return pd.NaT
    fecha = str(fecha).strip()
    
    # Caso: "Marzo 26" -> Asumir año 2026 del contexto
    if fecha.lower().startswith('marzo'):
        try:
            dia = fecha.split()[-1]
            return pd.Timestamp(f'2026-03-{int(dia):02d}')
        except:
            return pd.NaT
    
    # Intentar múltiples formatos
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y']:
        try:
            return pd.to_datetime(fecha, format=fmt)
        except:
            continue
    return pd.NaT

df_clean['Fecha_Control'] = df_clean['Fecha_Control'].apply(parsear_fecha)

fechas_fallidas = df_clean['Fecha_Control'].isna().sum()
print(f'Fechas que no pudieron parsearse: {fechas_fallidas}')
print(f'Rango de fechas: {df_clean["Fecha_Control"].min()} a {df_clean["Fecha_Control"].max()}')
print(f'\nTipo de dato resultante: {df_clean["Fecha_Control"].dtype}')

### 5.3 Corregir valores negativos en `Litros_Leche`

In [ ]:
# Convertir a valor absoluto (asumir error de signo)
negativos_antes = (df_clean['Litros_Leche'] < 0).sum()
df_clean['Litros_Leche'] = df_clean['Litros_Leche'].abs()
print(f'Valores negativos corregidos (abs): {negativos_antes}')

### 5.4 Tratar temperaturas anómalas

In [ ]:
# Reemplazar temperaturas claramente erróneas (>45°C) con NaN
mascara_temp = df_clean['Temp_Corporal'] > 45.0
print(f'Temperaturas >45°C reemplazadas por NaN: {mascara_temp.sum()}')
df_clean.loc[mascara_temp, 'Temp_Corporal'] = np.nan

print(f'Rango de temperatura resultante: [{df_clean["Temp_Corporal"].min()}, {df_clean["Temp_Corporal"].max()}]')

---
## 6. Eliminación de Duplicados y Dataset Limpio

In [ ]:
antes = len(df_clean)
df_clean = df_clean.drop_duplicates(keep='first')
despues = len(df_clean)
print(f'Registros antes: {antes}')
print(f'Registros después de eliminar duplicados: {despues}')
print(f'Duplicados eliminados: {antes - despues}')

In [ ]:
# Resumen final del dataset limpio
print('=== RESUMEN FINAL DEL DATASET LIMPIO ===')
print(f'Dimensiones: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas')
print(f'\nNulos restantes:')
print(df_clean.isnull().sum())
print(f'\nTipos de datos:')
print(df_clean.dtypes)
print(f'\nEstadísticas descriptivas:')
df_clean.describe()

In [ ]:
df_clean.info()

In [ ]:
df_clean.head(15)

---
## Resumen de Hallazgos

| Pilar | Hallazgo | Acción |
|-------|----------|--------|
| Valores nulos | `Litros_Leche` con valores NULL (string) | Parseados como NaN en la carga |
| Duplicados | Múltiples filas repetidas | Eliminadas con `drop_duplicates(keep='first')` |
| Inconsistencia de formato | `Fecha_Control` con 4 formatos distintos (ISO, DD/MM/YYYY, MM-DD-YYYY, texto) | Unificadas a `datetime64` |
| Inconsistencia categórica | `Estado_Salud` con variaciones de mayúsculas/minúsculas | Estandarizado a Title Case |
| Anomalías numéricas | `Litros_Leche` con valores negativos | Corregidos con `abs()` |
| Anomalías numéricas | `Temp_Corporal` con valores >45°C (errores de registro) | Reemplazados por NaN |